In [1]:
import pandas as pd
import numpy as np
import joblib, json
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

DATA_2025 = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/models/Crimes_-_2025_20260311.csv")   
SCHEMA_PATH = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/feature_schema.json")
LR_MODEL = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/models/best_logistic_regression.joblib")
RF_MODEL = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/models/best_random_forest.joblib")
LT_MODEL = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/models/chicago_crime_model_lightgbm_tuned.joblib")
XT_MODEL = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/models/chicago_crime_model_xgboost_tuned.joblib")

In [2]:
df_raw = pd.read_csv(DATA_2025, low_memory=False)
df_raw.columns = [c.strip() for c in df_raw.columns]

print(df_raw.columns.tolist())
print(df_raw.head(3))
print(df_raw.shape)

['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']
         ID Case Number                    Date                    Block  \
0  14075483    JK105557  12/31/2025 11:58:00 PM       050XX S PAULINA ST   
1  14070833    JK100050  12/31/2025 11:55:00 PM  053XX W WASHINGTON BLVD   
2  14070799    JK100014  12/31/2025 11:54:00 PM         100XX W OHARE ST   

   IUCR            Primary Type                    Description  \
0  0560                 ASSAULT                         SIMPLE   
1  0930     MOTOR VEHICLE THEFT  THEFT / RECOVERY - AUTOMOBILE   
2  2890  PUBLIC PEACE VIOLATION                OTHER VIOLATION   

  Location Description  Arrest  Domestic  ...  Ward  Community Area  FBI Code  \
0            RESIDENCE   False     False  ...  20.0            61.0       08

In [3]:
need_cols = ["Date","Primary Type","Description","Community Area","Location Description","Domestic","Arrest"]
missing = [c for c in need_cols if c not in df_raw.columns]
print("missing cols:", missing)

df = df_raw[need_cols].copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date","Community Area"])
df["Community Area"] = df["Community Area"].astype(int)

df["week_start"] = df["Date"].dt.to_period("W-SUN").dt.start_time
df["month"] = df["week_start"].dt.month
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype(int)

df["is_theft"] = (df["Primary Type"] == "THEFT").astype(int)
df["is_battery"] = (df["Primary Type"] == "BATTERY").astype(int)
df["is_burglary"] = (df["Primary Type"] == "BURGLARY").astype(int)

def top_loc_freq(x: pd.Series) -> float:
    x = x.dropna()
    if len(x) == 0:
        return 0.0
    vc = x.value_counts()
    return float(vc.iloc[0] / len(x))

weekly = (
    df.groupby(["Community Area","week_start"])
      .agg(
          total_crime_count=("Primary Type","size"),
          theft_count=("is_theft","sum"),
          battery_count=("is_battery","sum"),
          burglary_count=("is_burglary","sum"),
          arrest_rate=("Arrest","mean"),
          domestic_rate=("Domestic","mean"),
          top_location_freq=("Location Description", top_loc_freq),
          month=("month","first"),
          week_of_year=("week_of_year","first"),
      )
      .reset_index()
      .sort_values(["Community Area","week_start"])
)

g = weekly.groupby("Community Area")["total_crime_count"]
weekly["lag_1"] = g.shift(1)
weekly["lag_2"] = g.shift(2)
weekly["rolling_mean_4"] = g.shift(1).rolling(4).mean()
weekly["rolling_mean_8"] = g.shift(1).rolling(8).mean()

weekly["next_week_crime_count"] = g.shift(-1)
weekly["q75_next_week"] = weekly.groupby("week_start")["next_week_crime_count"].transform(lambda s: s.quantile(0.75))
weekly["is_high_risk_next_week"] = (weekly["next_week_crime_count"] >= weekly["q75_next_week"]).astype(int)

weekly = weekly.dropna(subset=["next_week_crime_count","lag_1","lag_2","rolling_mean_4","rolling_mean_8"]).reset_index(drop=True)

print(weekly.shape)
weekly.head()

missing cols: []


/var/folders/_3/28brymxx09lcz093mhsvp0240000gn/T/ipykernel_63630/2139797534.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


(3385, 18)


,Community Area,week_start,total_crime_count,theft_count,battery_count,burglary_count,arrest_rate,domestic_rate,top_location_freq,month,week_of_year,lag_1,lag_2,rolling_mean_4,rolling_mean_8,next_week_crime_count,q75_next_week,is_high_risk_next_week
0,1,2025-02-24,70,15,12,1,0.200000,0.171429,0.347826,2,9,88.0,67.0,77.50,71.875,75.0,72.00,1
1,1,2025-03-03,75,21,13,1,0.173333,0.173333,0.333333,3,10,70.0,88.0,76.25,74.875,67.0,80.00,0
2,1,2025-03-10,67,21,10,0,0.268657,0.164179,0.268657,3,11,75.0,70.0,75.00,75.875,79.0,68.00,1
3,1,2025-03-17,79,30,11,0,0.202532,0.075949,0.256410,3,12,67.0,75.0,75.00,75.500,74.0,78.25,0
4,1,2025-03-24,74,19,16,5,0.162162,0.135135,0.472973,3,13,79.0,67.0,72.75,75.125,67.0,73.00,0


In [4]:
with open(SCHEMA_PATH, "r") as f:
    schema = json.load(f)

feature_cols = schema["feature_columns"]
target_col = schema["target_column"]

missing_for_lr_rf = [c for c in feature_cols + [target_col] if c not in weekly.columns]
print("missing_for_lr_rf:", missing_for_lr_rf)

X_test = weekly[feature_cols].copy()
y_test = weekly[target_col].astype(int).copy()

print(X_test.shape, y_test.mean())

missing_for_lr_rf: []
(3385, 14) 0.26233382570162483


Lt & Xt

In [5]:
weekly_ltx = weekly.copy()

weekly_ltx["lag_crime_1"] = weekly_ltx["lag_1"]
weekly_ltx["lag_crime_2"] = weekly_ltx["lag_2"]

g2 = weekly_ltx.groupby("Community Area")["total_crime_count"]
weekly_ltx["lag_crime_4"] = g2.shift(4)

weekly_ltx = weekly_ltx.rename(columns={"Community Area": "Community_Area"})

weekly_ltx = weekly_ltx.dropna(subset=[
    "lag_crime_1", "lag_crime_2", "lag_crime_4", "rolling_mean_4", "next_week_crime_count"
]).reset_index(drop=True)

ltxt_feature_cols = [
    "Community_Area",
    "lag_crime_1",
    "lag_crime_2",
    "lag_crime_4",
    "rolling_mean_4",
    "month",
    "week_of_year",
]

missing_ltxt = [c for c in ltxt_feature_cols + [target_col] if c not in weekly_ltx.columns]
print("missing_ltxt:", missing_ltxt)

X_test_ltxt = weekly_ltx[ltxt_feature_cols].copy()
y_test_ltxt = weekly_ltx[target_col].astype(int).copy()

print(X_test_ltxt.shape, y_test_ltxt.mean())

missing_ltxt: []
(3077, 7) 0.262918427039324


In [6]:
lr = joblib.load(LR_MODEL)
rf = joblib.load(RF_MODEL)
lt = joblib.load(LT_MODEL)
xt = joblib.load(XT_MODEL)

print(type(lr))
print(type(rf))
print(type(lt))
print(type(xt))

<class 'sklearn.pipeline.Pipeline'>
<class 'sklearn.pipeline.Pipeline'>
<class 'lightgbm.sklearn.LGBMClassifier'>
<class 'xgboost.sklearn.XGBClassifier'>


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
def eval_model(model, X, y, thr=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "F1-score": f1_score(y, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y, proba),
    }

In [11]:
lt_feature_cols = [
    "Community_Area",
    "lag_crime_1",
    "lag_crime_2",
    "lag_crime_4",
    "rolling_mean_4",
    "month",
    "week_of_year",
]

xt_feature_cols = [
    "Community Area",
    "lag_crime_1",
    "lag_crime_2",
    "lag_crime_4",
    "rolling_mean_4",
    "month",
    "week_of_year",
]

# LT 用 Community_Area
X_test_lt = weekly_ltx[lt_feature_cols].copy()
y_test_lt = weekly_ltx[target_col].astype(int).copy()

# XT 用 Community Area
weekly_xt = weekly_ltx.rename(columns={"Community_Area": "Community Area"}).copy()
X_test_xt = weekly_xt[xt_feature_cols].copy()
y_test_xt = weekly_xt[target_col].astype(int).copy()

print("LT:", X_test_lt.shape, y_test_lt.mean())
print("XT:", X_test_xt.shape, y_test_xt.mean())

LT: (3077, 7) 0.262918427039324
XT: (3077, 7) 0.262918427039324


In [12]:
lr_metrics = eval_model(lr, X_test, y_test)
rf_metrics = eval_model(rf, X_test, y_test)

lt_metrics = eval_model(lt, X_test_lt, y_test_lt)
xt_metrics = eval_model(xt, X_test_xt, y_test_xt)

print("LR:", lr_metrics)
print("RF:", rf_metrics)
print("LT:", lt_metrics)
print("XT:", xt_metrics)

LR: {'accuracy': 0.9367799113737075, 'precision': 0.9680555555555556, 'recall': 0.7849099099099099, 'F1-score': 0.8669154228855721, 'ROC-AUC': 0.9843645708183153}
RF: {'accuracy': 0.9361890694239291, 'precision': 0.9759206798866855, 'recall': 0.7759009009009009, 'F1-score': 0.8644918444165621, 'ROC-AUC': 0.986011141297485}
LT: {'accuracy': 0.924276893077673, 'precision': 0.8295194508009154, 'recall': 0.896168108776267, 'F1-score': 0.8615567439096851, 'ROC-AUC': 0.983335349888708}
XT: {'accuracy': 0.9278518037049074, 'precision': 0.8385236447520185, 'recall': 0.8986402966625463, 'F1-score': 0.8675417661097852, 'ROC-AUC': 0.9833364399186401}


In [13]:
comparison_all = pd.DataFrame([
    {"model": "LogisticRegression", **lr_metrics, "n_test": len(X_test)},
    {"model": "RandomForest", **rf_metrics, "n_test": len(X_test)},
    {"model": "LightGBM", **lt_metrics, "n_test": len(X_test_lt)},
    {"model": "XGBoost", **xt_metrics, "n_test": len(X_test_xt)},
])

comparison_all

,model,accuracy,precision,recall,F1-score,ROC-AUC,n_test
0,LogisticRegression,0.936780,0.968056,0.784910,0.866915,0.984365,3385
1,RandomForest,0.936189,0.975921,0.775901,0.864492,0.986011,3385
2,LightGBM,0.924277,0.829519,0.896168,0.861557,0.983335,3077
3,XGBoost,0.927852,0.838524,0.898640,0.867542,0.983336,3077


In [14]:
out_path = Path("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/test_2025_all_models_comparison.csv")
comparison_all.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/test_2025_all_models_comparison.csv


**Spatial Evaluation**

In [15]:
# ===== LR / RF 预测结果 =====
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_pred = (lr_proba >= 0.5).astype(int)

rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_proba >= 0.5).astype(int)

spatial_lr_rf = weekly[["Community Area", "week_start"]].copy()
spatial_lr_rf["y_true"] = y_test.values
spatial_lr_rf["lr_pred"] = lr_pred
spatial_lr_rf["rf_pred"] = rf_pred
spatial_lr_rf["lr_proba"] = lr_proba
spatial_lr_rf["rf_proba"] = rf_proba

print(spatial_lr_rf.head())
print(spatial_lr_rf.shape)

   Community Area week_start  y_true  lr_pred  rf_pred  lr_proba  rf_proba
0               1 2025-02-24       1        0        0  0.287644  0.319470
1               1 2025-03-03       0        0        0  0.330679  0.291294
2               1 2025-03-10       1        0        0  0.231439  0.252387
3               1 2025-03-17       0        0        0  0.258967  0.283597
4               1 2025-03-24       0        0        0  0.283279  0.247974
(3385, 7)


In [16]:
lt_proba = lt.predict_proba(X_test_lt)[:, 1]
lt_pred = (lt_proba >= 0.5).astype(int)

xt_proba = xt.predict_proba(X_test_xt)[:, 1]
xt_pred = (xt_proba >= 0.5).astype(int)

# weekly_ltx 里区域列叫 Community_Area
spatial_ltxt = weekly_ltx[["Community_Area", "week_start"]].copy()
spatial_ltxt["y_true"] = y_test_lt.values
spatial_ltxt["lt_pred"] = lt_pred
spatial_ltxt["xt_pred"] = xt_pred
spatial_ltxt["lt_proba"] = lt_proba
spatial_ltxt["xt_proba"] = xt_proba

print(spatial_ltxt.head())
print(spatial_ltxt.shape)

   Community_Area week_start  y_true  lt_pred  xt_pred  lt_proba  xt_proba
0               1 2025-03-24       0        0        0  0.400821  0.470058
1               1 2025-03-31       1        0        1  0.431585  0.513660
2               1 2025-04-07       0        0        0  0.336815  0.467616
3               1 2025-04-14       0        1        1  0.525077  0.540808
4               1 2025-04-21       0        0        1  0.381064  0.516188
(3077, 7)


In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def area_metrics(df, area_col, y_col, pred_col):
    rows = []
    for area, g in df.groupby(area_col):
        y_true = g[y_col]
        y_pred = g[pred_col]
        rows.append({
            area_col: area,
            "n_samples": len(g),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "F1-score": f1_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)

In [ ]:
lr_area_metrics = area_metrics(spatial_lr_rf, "Community Area", "y_true", "lr_pred")
rf_area_metrics = area_metrics(spatial_lr_rf, "Community Area", "y_true", "rf_pred")
lt_area_metrics = area_metrics(spatial_ltxt, "Community_Area", "y_true", "lt_pred")
xt_area_metrics = area_metrics(spatial_ltxt, "Community_Area", "y_true", "xt_pred")


In [19]:
print("=== Logistic Regression: Top 5 Areas by F1 ===")
display(lr_area_metrics.head(5))
print("=== Logistic Regression: Bottom 5 Areas by F1 ===")
display(lr_area_metrics.tail(5))

print("=== Random Forest: Top 5 Areas by F1 ===")
display(rf_area_metrics.head(5))
print("=== Random Forest: Bottom 5 Areas by F1 ===")
display(rf_area_metrics.tail(5))

print("=== LightGBM: Top 5 Areas by F1 ===")
display(lt_area_metrics.head(5))
print("=== LightGBM: Bottom 5 Areas by F1 ===")
display(lt_area_metrics.tail(5))

print("=== XGBoost: Top 5 Areas by F1 ===")
display(xt_area_metrics.head(5))
print("=== XGBoost: Bottom 5 Areas by F1 ===")
display(xt_area_metrics.tail(5))

=== Logistic Regression: Top 5 Areas by F1 ===


,Community Area,n_samples,accuracy,precision,recall,F1-score
0,24,44,1.0,1.0,1.0,1.0
1,8,44,1.0,1.0,1.0,1.0
2,69,44,1.0,1.0,1.0,1.0
3,25,44,1.0,1.0,1.0,1.0
4,71,44,1.0,1.0,1.0,1.0


=== Logistic Regression: Bottom 5 Areas by F1 ===


,Community Area,n_samples,accuracy,precision,recall,F1-score
72,38,44,0.863636,0.0,0.0,0.0
73,2,44,0.954545,0.0,0.0,0.0
74,40,44,0.977273,0.0,0.0,0.0
75,41,44,1.000000,0.0,0.0,0.0
76,77,44,0.954545,0.0,0.0,0.0


=== Random Forest: Top 5 Areas by F1 ===


...

,Community_Area,n_samples,accuracy,precision,recall,F1-score
72,40,40,0.975,0.0,0.0,0.0
73,41,40,1.000,0.0,0.0,0.0
74,42,40,0.775,0.0,0.0,0.0
75,45,40,1.000,0.0,0.0,0.0
76,77,40,0.975,0.0,0.0,0.0


=== XGBoost: Top 5 Areas by F1 ===


,Community_Area,n_samples,accuracy,precision,recall,F1-score
0,25,40,1.0,1.0,1.0,1.0
1,43,40,1.0,1.0,1.0,1.0
2,24,40,1.0,1.0,1.0,1.0
3,23,40,1.0,1.0,1.0,1.0
4,28,40,1.0,1.0,1.0,1.0


=== XGBoost: Bottom 5 Areas by F1 ===


,Community_Area,n_samples,accuracy,precision,recall,F1-score
72,38,40,0.850,0.0,0.0,0.0
73,40,40,0.975,0.0,0.0,0.0
74,41,40,1.000,0.0,0.0,0.0
75,45,40,1.000,0.0,0.0,0.0
76,77,40,0.975,0.0,0.0,0.0


In [21]:
print("LR spatial mean F1:", lr_area_metrics["F1-score"].mean())
print("LR spatial std F1:", lr_area_metrics["F1-score"].std())

print("RF spatial mean F1:", rf_area_metrics["F1-score"].mean())
print("RF spatial std F1:", rf_area_metrics["F1-score"].std())

print("LT spatial mean F1:", lt_area_metrics["F1-score"].mean())
print("LT spatial std F1:", lt_area_metrics["F1-score"].std())

print("XT spatial mean F1:", xt_area_metrics["F1-score"].mean())
print("XT spatial std F1:", xt_area_metrics["F1-score"].std())

LR spatial mean F1: 0.21979827747376152
LR spatial std F1: 0.39795158732718994
RF spatial mean F1: 0.21179510508824828
RF spatial std F1: 0.3990455259323799
LT spatial mean F1: 0.26012502352918265
LT spatial std F1: 0.4098622142381699
XT spatial mean F1: 0.2701985483754348
XT spatial std F1: 0.4084938396017134


In [22]:
spatial_summary = pd.DataFrame([
    {"model": "LogisticRegression", "mean_area_F1": lr_area_metrics["F1-score"].mean(), "std_area_F1": lr_area_metrics["F1-score"].std()},
    {"model": "RandomForest", "mean_area_F1": rf_area_metrics["F1-score"].mean(), "std_area_F1": rf_area_metrics["F1-score"].std()},
    {"model": "LightGBM", "mean_area_F1": lt_area_metrics["F1-score"].mean(), "std_area_F1": lt_area_metrics["F1-score"].std()},
    {"model": "XGBoost", "mean_area_F1": xt_area_metrics["F1-score"].mean(), "std_area_F1": xt_area_metrics["F1-score"].std()},
])

spatial_summary
spatial_summary.to_csv("/Users/lijin/Desktop/Team8_IT5006_Predictive_Policing_AY2526Sem2/artifacts/spatial_summary_2025.csv", index=False)

**Temporal accuracy measures**

In [23]:
spatial_lr_rf["month"] = pd.to_datetime(spatial_lr_rf["week_start"]).dt.month
spatial_ltxt["month"] = pd.to_datetime(spatial_ltxt["week_start"]).dt.month

print(spatial_lr_rf[["Community Area", "week_start", "month"]].head())
print(spatial_ltxt[["Community_Area", "week_start", "month"]].head())

   Community Area week_start  month
0               1 2025-02-24      2
1               1 2025-03-03      3
2               1 2025-03-10      3
3               1 2025-03-17      3
4               1 2025-03-24      3
   Community_Area week_start  month
0               1 2025-03-24      3
1               1 2025-03-31      3
2               1 2025-04-07      4
3               1 2025-04-14      4
4               1 2025-04-21      4


In [24]:
def monthly_metrics(df, month_col, y_col, pred_col):
    rows = []
    for m, g in df.groupby(month_col):
        y_true = g[y_col]
        y_pred = g[pred_col]
        rows.append({
            "month": int(m),
            "n_samples": len(g),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "F1-score": f1_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows).sort_values("month").reset_index(drop=True)

In [25]:
lr_monthly = monthly_metrics(spatial_lr_rf, "month", "y_true", "lr_pred")
rf_monthly = monthly_metrics(spatial_lr_rf, "month", "y_true", "rf_pred")
lt_monthly = monthly_metrics(spatial_ltxt, "month", "y_true", "lt_pred")
xt_monthly = monthly_metrics(spatial_ltxt, "month", "y_true", "xt_pred")

print("LR monthly")
display(lr_monthly)

print("RF monthly")
display(rf_monthly)

print("LT monthly")
display(lt_monthly)

print("XT monthly")
display(xt_monthly)

LR monthly


,month,n_samples,precision,recall,F1-score
0,2,77,1.000000,0.750000,0.857143
1,3,384,0.987805,0.818182,0.895028
2,4,308,0.969697,0.800000,0.876712
3,5,308,1.000000,0.802469,0.890411
4,6,385,0.988764,0.871287,0.926316
5,7,307,0.945946,0.853659,0.897436
6,8,308,0.916667,0.814815,0.862745
7,9,385,0.939024,0.754902,0.836957
8,10,308,0.950820,0.716049,0.816901
9,11,308,0.984127,0.775000,0.867133


RF monthly


,month,n_samples,precision,recall,F1-score
0,2,77,1.000000,0.700000,0.823529
1,3,384,0.987654,0.808081,0.888889
2,4,308,0.969231,0.787500,0.868966
3,5,308,1.000000,0.753086,0.859155
4,6,385,1.000000,0.811881,0.896175
5,7,307,0.957143,0.817073,0.881579
6,8,308,0.955882,0.802469,0.872483
7,9,385,0.937500,0.735294,0.824176
8,10,308,0.967742,0.740741,0.839161
9,11,308,0.984127,0.775000,0.867133


LT monthly


,month,n_samples,precision,recall,F1-score
0,3,153,0.760870,0.875000,0.813953
1,4,308,0.783505,0.950000,0.858757
2,5,308,0.770833,0.913580,0.836158
3,6,385,0.855769,0.881188,0.868293
4,7,307,0.804124,0.951220,0.871508
5,8,308,0.820225,0.901235,0.858824
6,9,385,0.852941,0.852941,0.852941
7,10,308,0.865854,0.876543,0.871166
8,11,308,0.880952,0.925000,0.902439
9,12,307,0.883117,0.839506,0.860759


XT monthly


,month,n_samples,precision,recall,F1-score
0,3,153,0.857143,0.900000,0.878049
1,4,308,0.829545,0.912500,0.869048
2,5,308,0.822222,0.913580,0.865497
3,6,385,0.869159,0.920792,0.894231
4,7,307,0.762376,0.939024,0.841530
5,8,308,0.802198,0.901235,0.848837
6,9,385,0.812500,0.892157,0.850467
7,10,308,0.829545,0.901235,0.863905
8,11,308,0.900000,0.900000,0.900000
9,12,307,0.955882,0.802469,0.872483


In [27]:
temporal_summary = pd.DataFrame([
    {
        "model": "LogisticRegression",
        "mean_monthly_F1": lr_monthly["F1-score"].mean(),
        "std_monthly_F1": lr_monthly["F1-score"].std(),
        "mean_monthly_precision": lr_monthly["precision"].mean(),
        "mean_monthly_recall": lr_monthly["recall"].mean(),
    },
    {
        "model": "RandomForest",
        "mean_monthly_F1": rf_monthly["F1-score"].mean(),
        "std_monthly_F1": rf_monthly["F1-score"].std(),
        "mean_monthly_precision": rf_monthly["precision"].mean(),
        "mean_monthly_recall": rf_monthly["recall"].mean(),
    },
    {
        "model": "LightGBM",
        "mean_monthly_F1": lt_monthly["F1-score"].mean(),
        "std_monthly_F1": lt_monthly["F1-score"].std(),
        "mean_monthly_precision": lt_monthly["precision"].mean(),
        "mean_monthly_recall": lt_monthly["recall"].mean(),
    },
    {
        "model": "XGBoost",
        "mean_monthly_F1": xt_monthly["F1-score"].mean(),
        "std_monthly_F1": xt_monthly["F1-score"].std(),
        "mean_monthly_precision": xt_monthly["precision"].mean(),
        "mean_monthly_recall": xt_monthly["recall"].mean(),
    },
])

temporal_summary

,model,mean_monthly_F1,std_monthly_F1,mean_monthly_precision,mean_monthly_recall
0,LogisticRegression,0.863592,0.042615,0.971168,0.780545
1,RandomForest,0.861119,0.024516,0.978116,0.770170
2,LightGBM,0.859480,0.023253,0.827819,0.896621
3,XGBoost,0.868405,0.018947,0.844057,0.898299


**Robustness Analysis**

In [29]:
def eval_thresholds(model, X, y, model_name, thresholds=[0.3, 0.5, 0.7]):
    proba = model.predict_proba(X)[:, 1]
    rows = []
    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        rows.append({
            "model": model_name,
            "threshold": thr,
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "F1-score": f1_score(y, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

In [30]:
lr_thr = eval_thresholds(lr, X_test, y_test, "LogisticRegression")
rf_thr = eval_thresholds(rf, X_test, y_test, "RandomForest")

lt_thr = eval_thresholds(lt, X_test_lt, y_test_lt, "LightGBM")
xt_thr = eval_thresholds(xt, X_test_xt, y_test_xt, "XGBoost")

robustness_df = pd.concat([lr_thr, rf_thr, lt_thr, xt_thr], ignore_index=True)
robustness_df

,model,threshold,precision,recall,F1-score
0,LogisticRegression,0.3,0.901639,0.867117,0.884041
1,LogisticRegression,0.5,0.968056,0.784910,0.866915
2,LogisticRegression,0.7,0.990521,0.706081,0.824458
3,RandomForest,0.3,0.912784,0.860360,0.885797
4,RandomForest,0.5,0.975921,0.775901,0.864492
5,RandomForest,0.7,0.993600,0.699324,0.820886
6,LightGBM,0.3,0.759881,0.950556,0.844591
7,LightGBM,0.5,0.829519,0.896168,0.861557
8,LightGBM,0.7,0.948905,0.803461,0.870147
9,XGBoost,0.3,0.262918,1.000000,0.416366


In [31]:
print("=== Logistic Regression ===")
display(lr_thr)

print("=== Random Forest ===")
display(rf_thr)

print("=== LightGBM ===")
display(lt_thr)

print("=== XGBoost ===")
display(xt_thr)

=== Logistic Regression ===


,model,threshold,precision,recall,F1-score
0,LogisticRegression,0.3,0.901639,0.867117,0.884041
1,LogisticRegression,0.5,0.968056,0.784910,0.866915
2,LogisticRegression,0.7,0.990521,0.706081,0.824458


=== Random Forest ===


,model,threshold,precision,recall,F1-score
0,RandomForest,0.3,0.912784,0.860360,0.885797
1,RandomForest,0.5,0.975921,0.775901,0.864492
2,RandomForest,0.7,0.993600,0.699324,0.820886


=== LightGBM ===


,model,threshold,precision,recall,F1-score
0,LightGBM,0.3,0.759881,0.950556,0.844591
1,LightGBM,0.5,0.829519,0.896168,0.861557
2,LightGBM,0.7,0.948905,0.803461,0.870147


=== XGBoost ===


,model,threshold,precision,recall,F1-score
0,XGBoost,0.3,0.262918,1.00000,0.416366
1,XGBoost,0.5,0.838524,0.89864,0.867542
2,XGBoost,0.7,0.000000,0.00000,0.000000
